In [0]:
-- =============================================================================
-- SILVER LAYER — Retail Sales Data Warehouse
-- Catalog : retail_sales_catalog  |  Schema : silver
--
-- FIX APPLIED: Replaced INSERT INTO with MERGE on DimProduct, DimStore,
-- and FactSales so re-running the notebook never creates duplicate rows.
-- DimCustomer already used MERGE — no change needed there.
-- =============================================================================

USE CATALOG retail_sales_catalog;
USE SCHEMA silver;


-- =============================================================================
-- TABLE 1 : DimCustomer  (SCD Type 2)
-- No change needed here — MERGE is already idempotent (safe to re-run)
-- =============================================================================

CREATE TABLE IF NOT EXISTS retail_sales_catalog.silver.DimCustomer (
    CustomerSK   BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID   INT,
    CustomerName STRING,
    Email        STRING,
    City         STRING,
    Address      STRING,
    StartDate    DATE,
    EndDate      DATE,
    IsActive     INT
)
USING DELTA LOCATION 's3://retail-dwh-capstone-project/processed/dim_customer/';


-- STEP 2A: Expire old active records where City or Address changed
MERGE INTO retail_sales_catalog.silver.DimCustomer AS target
USING (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName))  AS CustomerName,
        LOWER(TRIM(Email))           AS Email,
        TRIM(City)                   AS City,
        TRIM(Address)                AS Address
    FROM (
        SELECT
            CustomerID,
            CustomerName,
            Email,
            City,
            Address,
            LastUpdated,
            ROW_NUMBER() OVER (
                PARTITION BY CustomerID
                ORDER BY LastUpdated DESC
            ) AS rn
        FROM retail_sales_catalog.bronze.customers_raw
        WHERE CustomerID IS NOT NULL
    )
    WHERE rn = 1
) AS source
ON  target.CustomerID = source.CustomerID
AND target.IsActive   = 1

WHEN MATCHED AND (
    target.City    <> source.City OR
    target.Address <> source.Address
)
THEN UPDATE SET
    target.EndDate  = CURRENT_DATE(),
    target.IsActive = 0

WHEN NOT MATCHED
THEN INSERT (CustomerID, CustomerName, Email, City, Address, StartDate, EndDate, IsActive)
VALUES (
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE(),
    DATE('9999-12-31'),
    1
);


-- STEP 2B: Insert new active version for customers that were just expired
INSERT INTO retail_sales_catalog.silver.DimCustomer (
    CustomerID, CustomerName, Email, City, Address, StartDate, EndDate, IsActive
)
SELECT
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE()     AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1                  AS IsActive
FROM (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName))  AS CustomerName,
        LOWER(TRIM(Email))           AS Email,
        TRIM(City)                   AS City,
        TRIM(Address)                AS Address,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY LastUpdated DESC
        ) AS rn
    FROM retail_sales_catalog.bronze.customers_raw
    WHERE CustomerID IS NOT NULL
) source
WHERE rn = 1
  AND NOT EXISTS (
      SELECT 1 FROM retail_sales_catalog.silver.DimCustomer t
      WHERE t.CustomerID = source.CustomerID
        AND t.IsActive   = 1
        AND t.City       = source.City
        AND t.Address    = source.Address
  )
  AND EXISTS (
      SELECT 1 FROM retail_sales_catalog.silver.DimCustomer t
      WHERE t.CustomerID = source.CustomerID
  );


-- Verify DimCustomer
SELECT COUNT(*)           AS total_rows  FROM retail_sales_catalog.silver.DimCustomer;
SELECT IsActive, COUNT(*) AS count       FROM retail_sales_catalog.silver.DimCustomer GROUP BY IsActive;
SELECT *                                 FROM retail_sales_catalog.silver.DimCustomer LIMIT 10;


-- =============================================================================
-- TABLE 2 : DimProduct
-- FIX: Replaced INSERT INTO with MERGE on ProductID
--      → re-running will never insert duplicate products
-- =============================================================================

CREATE TABLE IF NOT EXISTS retail_sales_catalog.silver.DimProduct (
    ProductSK     BIGINT,
    ProductID     INT,
    ProductName   STRING,
    Category      STRING,
    UnitPrice     DOUBLE,      -- changed INT → DOUBLE (prices can be decimal)
    EffectiveDate DATE
)
USING DELTA LOCATION 's3://retail-dwh-capstone-project/processed/dim_product/';


MERGE INTO retail_sales_catalog.silver.DimProduct AS target
USING (
    SELECT
        ProductID,
        TRIM(ProductName)  AS ProductName,
        TRIM(Category)     AS Category,
        UnitPrice,
        CURRENT_DATE()     AS EffectiveDate
    FROM retail_sales_catalog.bronze.products_raw
    WHERE ProductID IS NOT NULL
      AND UnitPrice > 0              -- reject zero-price records
) AS source
ON target.ProductID = source.ProductID    -- match on natural key

-- If product already exists → do nothing (no update needed, prices are static)
WHEN NOT MATCHED
THEN INSERT (ProductSK, ProductID, ProductName, Category, UnitPrice, EffectiveDate)
VALUES (
    monotonically_increasing_id(),
    source.ProductID,
    source.ProductName,
    source.Category,
    source.UnitPrice,
    source.EffectiveDate
);


-- Verify DimProduct
SELECT COUNT(*) AS total_products FROM retail_sales_catalog.silver.DimProduct;
SELECT * FROM retail_sales_catalog.silver.DimProduct LIMIT 5;


-- =============================================================================
-- TABLE 3 : DimStore
-- FIX: Replaced INSERT INTO with MERGE on StoreID
--      → re-running will never insert duplicate stores
-- =============================================================================

CREATE TABLE IF NOT EXISTS retail_sales_catalog.silver.DimStore (
    StoreSK   BIGINT,
    StoreID   INT,
    StoreName STRING,
    Region    STRING
)
USING DELTA LOCATION 's3://retail-dwh-capstone-project/processed/dim_stores/';


MERGE INTO retail_sales_catalog.silver.DimStore AS target
USING (
    SELECT
        StoreID,
        TRIM(INITCAP(StoreName))          AS StoreName,   -- fix casing
        COALESCE(TRIM(Region), 'Unknown') AS Region       -- NULL → 'Unknown'
    FROM retail_sales_catalog.bronze.stores_raw
    WHERE StoreID IS NOT NULL
) AS source
ON target.StoreID = source.StoreID        -- match on natural key

WHEN NOT MATCHED
THEN INSERT (StoreSK, StoreID, StoreName, Region)
VALUES (
    monotonically_increasing_id(),
    source.StoreID,
    source.StoreName,
    source.Region
);


-- Verify DimStore
SELECT COUNT(*) AS total_stores FROM retail_sales_catalog.silver.DimStore;
SELECT * FROM retail_sales_catalog.silver.DimStore LIMIT 5;


-- =============================================================================
-- TABLE 4 : FactSales
-- FIX: Replaced INSERT INTO with MERGE on TransactionID
--      → re-running will never insert duplicate transactions
-- Also handles:
--   - Duplicate TransactionIDs in source (ROW_NUMBER keeps earliest)
--   - Quantity = 0 records rejected
--   - Orphan CustomerIDs excluded (INNER JOIN on DimCustomer)
--   - Amount derived as Quantity × UnitPrice
-- =============================================================================

CREATE TABLE IF NOT EXISTS retail_sales_catalog.silver.FactSales (
    SalesSK       BIGINT,
    TransactionID INT,
    CustomerSK    BIGINT,
    ProductSK     BIGINT,
    StoreSK       BIGINT,
    Quantity      INT,
    Amount        DECIMAL(10,2),
    TxnDate       DATE
)
USING DELTA LOCATION 's3://retail-dwh-capstone-project/processed/dim_sales/';


MERGE INTO retail_sales_catalog.silver.FactSales AS target
USING (
    SELECT
        monotonically_increasing_id()      AS SalesSK,
        s.TransactionID,
        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,
        s.Quantity,
        ROUND(s.Quantity * p.UnitPrice, 2) AS Amount,
        DATE(s.TxnDate)                    AS TxnDate
    FROM (
        -- Deduplicate TransactionIDs — keep the earliest occurrence
        SELECT
            TransactionID, CustomerID, ProductID, StoreID, Quantity, TxnDate,
            ROW_NUMBER() OVER (
                PARTITION BY TransactionID
                ORDER BY TxnDate ASC     -- earliest record wins
            ) AS rn
        FROM retail_sales_catalog.bronze.sales_raw
        WHERE TransactionID IS NOT NULL
          AND Quantity > 0               -- reject zero-quantity rows
    ) s
    INNER JOIN retail_sales_catalog.silver.DimCustomer c
        ON  s.CustomerID = c.CustomerID
        AND c.IsActive   = 1             -- join to current active customer only
                                         -- this also excludes orphan CustomerIDs
    LEFT JOIN retail_sales_catalog.silver.DimProduct p
        ON s.ProductID = p.ProductID
    LEFT JOIN retail_sales_catalog.silver.DimStore st
        ON s.StoreID = st.StoreID
    WHERE s.rn = 1                       -- deduplicated transactions only
) AS source
ON target.TransactionID = source.TransactionID   -- match on natural key

WHEN NOT MATCHED
THEN INSERT (SalesSK, TransactionID, CustomerSK, ProductSK, StoreSK, Quantity, Amount, TxnDate)
VALUES (
    source.SalesSK,
    source.TransactionID,
    source.CustomerSK,
    source.ProductSK,
    source.StoreSK,
    source.Quantity,
    source.Amount,
    source.TxnDate
);


-- Verify FactSales
SELECT COUNT(*) AS total_sales FROM retail_sales_catalog.silver.FactSales;
SELECT * FROM retail_sales_catalog.silver.FactSales LIMIT 5;


-- =============================================================================
-- FINAL ROW COUNT SUMMARY
-- Expected (after truncate + clean run):
--   DimCustomer → 490  (488 active + 2 expired SCD2 records)
--   DimProduct  → 990  (1000 source - 10 with UnitPrice=0)
--   DimStore    → 1000
--   FactSales   → 2420 (after dedup + Qty>0 + orphan exclusion)
-- =============================================================================

SELECT 'DimCustomer' AS table_name, COUNT(*) AS row_count FROM retail_sales_catalog.silver.DimCustomer
UNION ALL
SELECT 'DimProduct',  COUNT(*) FROM retail_sales_catalog.silver.DimProduct
UNION ALL
SELECT 'DimStore',    COUNT(*) FROM retail_sales_catalog.silver.DimStore
UNION ALL
SELECT 'FactSales',   COUNT(*) FROM retail_sales_catalog.silver.FactSales;
SELECT
    IsActive,
    COUNT(*) AS count,
    CASE WHEN IsActive = 1 THEN 'Current active'
         ELSE 'Expired - SCD2 history'
    END AS meaning
FROM retail_sales_catalog.silver.DimCustomer
GROUP BY IsActive;


-- TRUNCATE TABLE retail_sales_catalog.silver.DimCustomer;

-- TRUNCATE TABLE retail_sales_catalog.silver.DimProduct;

-- TRUNCATE TABLE retail_sales_catalog.silver.DimStore;
-- TRUNCATE TABLE retail_sales_catalog.silver.factsales;